# Self-RAG Inference Generator

Objective: run the new standalone Self-RAG answer-generation pipeline over MS MARCO. This notebook is the local workflow for data inspection, FAISS index building, retrieval preview, and optional small generation checks.

The existing `06_self_rag_verifier*.ipynb` notebooks are intentionally left unchanged.

In [ ]:
import json
import logging
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "experiments"))

from rag_filtering.config.loader import load_yaml, resolve_path
from self_rag_inference.msmarco_corpus import MSMARCORetriever, load_msmarco_corpus

logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")
CONFIG_PATH = "configs/experiments/self_rag_inference.yaml"
RUN_GENERATION = False  # Use Kaggle T4 x2 for the full 7B Self-RAG run.
LIMIT = 5

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
cfg = load_yaml(CONFIG_PATH)
cfg["data"]["max_samples"] = LIMIT

print("Model:", cfg["model"]["name"])
print("Backend:", cfg["model"]["backend"])
print("MS MARCO CSV:", cfg["data"]["msmarco_csv"])
print("Top-k:", cfg["retriever"]["top_k"])
print("Results:", cfg["evaluation"]["results_dir"])

In [ ]:
corpus = load_msmarco_corpus(cfg["data"])
print(f"Rows: {len(corpus.rows)}")
print(f"Unique passages: {len(corpus.documents)}")

retriever = MSMARCORetriever(cfg=cfg["retriever"], corpus=corpus)
retriever.build_or_load(force_rebuild=False)

In [ ]:
preview_rows = []
for row in corpus.rows:
    passages = retriever.retrieve(row.question)
    preview_rows.append({
        "id": row.row_id,
        "question": row.question,
        "gold_answer": row.gold_answer[:160],
        "top_context": passages[0]["text"][:240] if passages else "",
        "score": passages[0]["score"] if passages else None,
    })

pd.DataFrame(preview_rows)

In [ ]:
if RUN_GENERATION:
    from self_rag_inference.run_inference import run_generation

    metrics = run_generation(cfg, corpus, retriever, top_k=cfg["retriever"]["top_k"])
    print(json.dumps(metrics, indent=2))
else:
    print("Generation skipped. Use notebooks/07_self_rag_inference_kaggle.ipynb for the full T4 x2 run.")

In [ ]:
results_dir = resolve_path(cfg["evaluation"]["results_dir"])
predictions_path = results_dir / cfg["evaluation"]["predictions_csv"]
metrics_path = results_dir / cfg["evaluation"]["metrics_json"]

if predictions_path.exists():
    display(pd.read_csv(predictions_path).head())
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text(encoding="utf-8")), indent=2))